# Partie 6 - Analyse de graphes avec GraphFrames (PySpark) et GraphX (Scala)
Graphe **comptes - appareils - beneficiaires** construit a partir de la couche Silver.

| Sommet | id | Arete | Sens | Attributs |
|---|---|---|---|---|
| `wallet` | `W-xxxxx` | `USES_DEVICE` | wallet -> device | nb_tx, montant_total_mad, premiere/derniere date |
| `device` | `D-xxxxxx` | `PAYS` | wallet -> merchant | nb_tx, montant_total_mad (COMPLETED, P2M) |
| `merchant` | `M-xxx` | `TRANSFERS_TO` | wallet -> wallet | nb_tx, montant_total_mad (COMPLETED, P2P) |

Les prefixes `W-`, `D-`, `M-` rendent les identifiants uniques et stables quel que soit le type.
Rappel : les indicateurs ci-dessous signalent des **structures inhabituelles** a examiner par
l'equipe conformite ; ils ne constituent pas une preuve automatique de fraude.

In [ ]:
import os, sys, json, time, subprocess
from functools import reduce
sys.path.insert(0, "/home/sahlpay/jobs"); sys.path.insert(0, "../jobs")
from sahlpay_common import get_spark, Paths, DATA_DIR, LOG_DIR, PROJECT_DIR, MERCHANT_SCHEMA, merchants_path, timer
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.utils import AnalysisException

spark = get_spark("SahlPay-P6-GraphFrames")
sc = spark.sparkContext
sc.setCheckpointDir(f"{Paths.checkpoints}/graphframes")   # requis par certains algorithmes GraphFrames
timings = {}
GRAPH_SOURCE = os.environ.get("GRAPH_SOURCE", "both")    # history | stream | both

## 1. Preparation des donnees : sommets et aretes a partir de la couche Silver

In [ ]:
cols = ["tx_id", "type", "payer_wallet", "payee_id", "device_id", "city", "status", "amount_mad", "event_time"]
sources = {"history": Paths.silver_history, "stream": Paths.silver_clean}
frames = []
for name, path in sources.items():
    if GRAPH_SOURCE in (name, "both"):
        try:
            df = spark.read.parquet(path).select(*cols)
            print(f"source {name:<8} {path} : {df.count()} evenements")
            frames.append(df)
        except AnalysisException:
            print(f"source {name:<8} {path} : absente (ignoree)")
tx = reduce(DataFrame.unionByName, frames).dropDuplicates(["tx_id", "status"]).cache()
print("evenements utilises :", tx.count())

one_per_tx = tx.dropDuplicates(["tx_id"])                                   # 1 ligne par transaction
completed = tx.filter(F.col("status") == "COMPLETED").dropDuplicates(["tx_id"])
merchants = spark.read.schema(MERCHANT_SCHEMA).json(merchants_path())

# --- Sommets ---
wallet_v = (one_per_tx.select(F.col("payer_wallet").alias("id"), "city")
            .unionByName(one_per_tx.filter(F.col("type") == "P2P").select(F.col("payee_id").alias("id"), "city"))
            .groupBy("id").agg(F.first("city").alias("city"))
            .select("id", F.lit("wallet").alias("type"), F.lit(None).cast("string").alias("name"),
                    "city", F.lit(None).cast("string").alias("category")))
device_v = (one_per_tx.select(F.col("device_id").alias("id")).distinct()
            .select("id", F.lit("device").alias("type"), F.lit(None).cast("string").alias("name"),
                    F.lit(None).cast("string").alias("city"), F.lit(None).cast("string").alias("category")))
merchant_v = merchants.select(F.col("merchant_id").alias("id"), F.lit("merchant").alias("type"),
                              "name", "city", "category")
vertices = wallet_v.unionByName(device_v).unionByName(merchant_v).dropDuplicates(["id"]).cache()

# --- Aretes (agregees : une arete par couple et par relation) ---
def agg_edges(df, src, dst, rel):
    return (df.groupBy(F.col(src).alias("src"), F.col(dst).alias("dst"))
              .agg(F.count("*").cast("long").alias("nb_tx"),
                   F.round(F.sum("amount_mad"), 2).cast("double").alias("montant_total_mad"),
                   F.min("event_time").alias("premiere_tx"),
                   F.max("event_time").alias("derniere_tx"))
              .withColumn("relationship", F.lit(rel)))

uses_device = agg_edges(one_per_tx, "payer_wallet", "device_id", "USES_DEVICE")
pays = agg_edges(completed.filter(F.col("type") == "P2M"), "payer_wallet", "payee_id", "PAYS")
transfers = agg_edges(completed.filter((F.col("type") == "P2P") & (F.col("payer_wallet") != F.col("payee_id"))),
                      "payer_wallet", "payee_id", "TRANSFERS_TO")
edges_all = uses_device.unionByName(pays).unionByName(transfers)

# Exclusion des aretes orphelines (une extremite absente des sommets)
ids = vertices.select("id")
edges = (edges_all.join(ids.withColumnRenamed("id", "src"), "src", "left_semi")
                  .join(ids.withColumnRenamed("id", "dst"), "dst", "left_semi")
                  .select("src", "dst", "relationship", "nb_tx", "montant_total_mad", "premiere_tx", "derniere_tx")
                  .cache())
n_all, n_kept = edges_all.count(), edges.count()
print(f"aretes construites : {n_all} | orphelines exclues : {n_all - n_kept} | aretes conservees : {n_kept}")

# Export Parquet (entrees communes GraphFrames / GraphX)
vertices.write.mode("overwrite").parquet(f"{Paths.silver_graph}/vertices")
edges.write.mode("overwrite").parquet(f"{Paths.silver_graph}/edges")
vertices = spark.read.parquet(f"{Paths.silver_graph}/vertices").cache()
edges = spark.read.parquet(f"{Paths.silver_graph}/edges").cache()

## 2. GraphFrame : chargement et controles

In [ ]:
from graphframes import GraphFrame

g = GraphFrame(vertices, edges)
print("sommets par type :"); g.vertices.groupBy("type").count().orderBy("type").show()
print("aretes par type :");  g.edges.groupBy("relationship").count().orderBy("relationship").show()
orph_src = g.edges.join(g.vertices.select(F.col("id").alias("src")), "src", "left_anti").count()
orph_dst = g.edges.join(g.vertices.select(F.col("id").alias("dst")), "dst", "left_anti").count()
print(f"aretes orphelines : src={orph_src} dst={orph_dst}")
assert orph_src == 0 and orph_dst == 0
print("schema des sommets :"); g.vertices.printSchema()
print("schema des aretes :");  g.edges.printSchema()

# Sous-graphes utilises par les motifs (eviter l'explosion combinatoire autour des gros commercants)
g_dev = GraphFrame(vertices, edges.filter(F.col("relationship") == "USES_DEVICE"))
g_tr = GraphFrame(vertices.filter(F.col("type") == "wallet"), edges.filter(F.col("relationship") == "TRANSFERS_TO"))
g_pay = GraphFrame(vertices, edges.filter(F.col("relationship").isin("TRANSFERS_TO", "PAYS")))

## 3. Motifs avec GraphFrames (`find`) et degres
### 3.a Appareils partages par plusieurs comptes : `(a)-[]->(d); (b)-[]->(d)`

In [ ]:
with timer("gf_motif_shared_device", timings):
    pairs = (g_dev.find("(a)-[e1]->(d); (b)-[e2]->(d)")
             .filter((F.col("d.type") == "device") & (F.col("a.id") < F.col("b.id"))))
    shared = (pairs.groupBy(F.col("d.id").alias("device_id"))
              .agg(F.array_sort(F.array_distinct(F.concat(F.collect_set("a.id"), F.collect_set("b.id")))).alias("comptes"),
                   F.count("*").alias("nb_paires"))
              .withColumn("nb_comptes", F.size("comptes"))
              .cache())
    n_shared = shared.count()
print("appareils utilises par >= 2 comptes :", n_shared)
shared.groupBy("nb_comptes").count().orderBy("nb_comptes").show()
shared.orderBy(F.desc("nb_comptes")).show(8, truncate=80)

### 3.b Transferts circulaires : `(a)-[]->(b); (b)-[]->(c); (c)-[]->(a)`
`a.id < b.id AND a.id < c.id` garde une seule rotation de chaque cycle oriente.

In [ ]:
with timer("gf_motif_cycles", timings):
    cycles = (g_tr.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)")
              .filter((F.col("a.id") < F.col("b.id")) & (F.col("a.id") < F.col("c.id")))
              .select(F.col("a.id").alias("a"), F.col("b.id").alias("b"), F.col("c.id").alias("c"),
                      F.least("e1.nb_tx", "e2.nb_tx", "e3.nb_tx").alias("min_nb_tx"),
                      F.least("e1.montant_total_mad", "e2.montant_total_mad", "e3.montant_total_mad").alias("min_montant_arete"),
                      (F.col("e1.montant_total_mad") + F.col("e2.montant_total_mad") + F.col("e3.montant_total_mad"))
                      .alias("montant_cycle"))
              .cache())
    n_cycles = cycles.count()
print("cycles de longueur 3 (tous montants) :", n_cycles)
cycles.orderBy(F.desc("montant_cycle")).show(10, truncate=False)

### 3.c Chaines compte -> compte -> commercant : `(a)-[e1]->(b); (b)-[e2]->(m)`

In [ ]:
with timer("gf_motif_chains", timings):
    chains = (g_pay.find("(a)-[e1]->(b); (b)-[e2]->(m)")
              .filter((F.col("e1.relationship") == "TRANSFERS_TO") & (F.col("e2.relationship") == "PAYS")
                      & (F.col("m.type") == "merchant"))
              .select(F.col("a.id").alias("emetteur"), F.col("b.id").alias("intermediaire"),
                      F.col("m.id").alias("commercant"), F.col("m.category").alias("categorie"),
                      F.col("e1.montant_total_mad").alias("montant_recu"),
                      F.col("e2.montant_total_mad").alias("montant_depense"),
                      ((F.col("e2.derniere_tx").cast("long") - F.col("e1.premiere_tx").cast("long")) / 3600)
                      .alias("ecart_heures"))
              .cache())
    n_chains = chains.count()
print("chaines compte -> compte -> commercant :", n_chains)
print("intermediaires recevant de nombreux emetteurs puis payant des commercants :")
(chains.groupBy("intermediaire")
       .agg(F.countDistinct("emetteur").alias("nb_emetteurs"), F.countDistinct("commercant").alias("nb_commercants"),
            F.round(F.sum("montant_depense"), 2).alias("montant_depense"))
       .orderBy(F.desc("nb_emetteurs")).show(8))

### 3.d Degres (graphe des transferts) et comptes collecteurs

In [ ]:
with timer("gf_degrees", timings):
    gf_in = g_tr.inDegrees.cache(); gf_out = g_tr.outDegrees.cache()
    gf_in.count(); gf_out.count()
received = (g_tr.edges.groupBy(F.col("dst").alias("id"))
            .agg(F.countDistinct("src").alias("nb_emetteurs"),
                 F.sum("nb_tx").alias("nb_virements_recus"),
                 F.round(F.sum("montant_total_mad"), 2).alias("montant_recu_mad"))
            .withColumn("montant_moyen_recu", F.round(F.col("montant_recu_mad") / F.col("nb_virements_recus"), 2)))
gf_deg = (gf_in.join(gf_out, "id", "full_outer").na.fill(0)
               .join(received, "id", "left").cache())
gf_deg.orderBy(F.desc("inDegree")).show(10)
gf_deg.select(F.max("inDegree"), F.expr("percentile_approx(inDegree, 0.99)").alias("p99_inDegree"),
              F.max("outDegree")).show()

## 4. GraphX (Scala) sur les memes tables
Le traitement Scala lit `lake/silver/graph/{vertices,edges}`, attribue un `VertexId` (Long) unique,
construit `Graph(verticesRDD, edgesRDD)` et calcule `degrees`, `connectedComponents` et `pageRank`
sur le graphe des transferts. Il tourne comme une **seconde application** sur le cluster
(2 coeurs chacune), on peut donc le lancer sans fermer cette session. Equivalent terminal :
`docker compose exec jupyter bash graphx/run_graphx.sh`.

In [ ]:
t0 = time.perf_counter()
run = subprocess.run(["bash", os.path.join(PROJECT_DIR, "graphx", "run_graphx.sh")],
                     capture_output=True, text=True, env={**os.environ, "LOG_DIR": LOG_DIR})
timings["graphx_total_wallclock"] = round(time.perf_counter() - t0, 2)
print(run.stdout[-6000:])
if run.returncode != 0:
    print(run.stderr[-3000:])
    raise RuntimeError("echec du traitement GraphX")

In [ ]:
gx = spark.read.parquet(f"{Paths.gold_graphx}/wallet_metrics").cache()
gx_timings = spark.read.parquet(f"{Paths.gold_graphx}/timings")
gx_full = spark.read.parquet(f"{Paths.gold_graphx}/full_degrees")
gx_timings.show(truncate=False)
gx.orderBy(F.desc("pagerank")).show(10)

## 5. Comparaison GraphX / GraphFrames sur le meme sous-graphe (transferts)

In [ ]:
cmp = (gf_deg.select("id", F.col("inDegree").alias("gf_in"), F.col("outDegree").alias("gf_out"))
       .join(gx.filter("degree > 0").select("id", F.col("in_degree").alias("gx_in"), F.col("out_degree").alias("gx_out")),
             "id", "full_outer").na.fill(0))
n_cmp = cmp.count()
n_diff = cmp.filter("gf_in != gx_in OR gf_out != gx_out").count()
print(f"wallets compares : {n_cmp} | differences de degres : {n_diff}")
cmp.orderBy(F.desc("gf_in")).show(10)

# meme comparaison sur le graphe complet (3 relations)
with timer("gf_degrees_full", timings):
    gf_full = g.inDegrees.join(g.outDegrees, "id", "full_outer").na.fill(0).cache()
    gf_full.count()
cmp_full = gf_full.join(gx_full.select("id", "in_degree", "out_degree"), "id", "full_outer").na.fill(0)
print("differences sur le graphe complet :",
      cmp_full.filter("inDegree != in_degree OR outDegree != out_degree").count(), "/", cmp_full.count())

gx_t = {r["etape"]: r["secondes"] for r in gx_timings.collect()}
print(f"\n{'Mesure':<36}{'GraphFrames (s)':>16}{'GraphX (s)':>12}")
print(f"{'degres (in + out), graphe transferts':<36}{timings['gf_degrees']:>16.2f}"
      f"{gx_t.get('inDegrees', 0) + gx_t.get('outDegrees', 0):>12.2f}")
print(f"{'degres, graphe complet':<36}{timings['gf_degrees_full']:>16.2f}{gx_t.get('degrees (graphe complet)', 0):>12.2f}")
print(f"{'motif appareils partages':<36}{timings['gf_motif_shared_device']:>16.2f}{'-':>12}")
print(f"{'motif cycles':<36}{timings['gf_motif_cycles']:>16.2f}{'-':>12}")
print(f"{'connectedComponents':<36}{'-':>16}{gx_t.get('connectedComponents', 0):>12.2f}")
print(f"{'pageRank':<36}{'-':>16}{gx_t.get('pageRank(tol=1e-4)', 0):>12.2f}")

**A commenter** : les degres sont identiques (meme graphe, aretes agregees par couple). GraphX
travaille sur des RDD types (`Graph[VD, ED]`, VertexId `Long`) en Scala : algorithmes iteratifs
(Pregel) integres, controle fin du partitionnement, mais il faut convertir les identifiants
metier en `Long`. GraphFrames manipule des DataFrames (ids `string`, colonnes nommees), profite de
Catalyst et propose la recherche de motifs (`find`) qui n'existe pas dans GraphX ; ses algorithmes
(PageRank, composantes) s'appuient en partie sur GraphX en interne. La **centralite** (PageRank,
degre entrant) mesure la position d'un compte dans le reseau : un compte central peut etre un
commercant informel, un proprietaire ou une tontine legitime. Elle oriente l'investigation,
elle ne prouve pas une fraude.

## 6. Identification des anomalies : seuils explicites

In [ ]:
SEUILS = {
    "SHARED_DEVICE": {"nb_comptes_par_appareil_min": 5},
    "COLLECTOR_ACCOUNT": {"nb_emetteurs_distincts_min": 15, "montant_moyen_recu_max_mad": 100},
    "CIRCULAR_TRANSFER": {"longueur": 3, "nb_tx_min_par_arete": 2, "montant_min_par_arete_mad": 1000},
    "LARGE_COMPONENT": {"taille_composante_min": 40},
}
print(json.dumps(SEUILS, indent=2, ensure_ascii=False))
S = SEUILS
now = F.current_timestamp()

# 6.1 appareils partages
an_shared = (shared.filter(F.col("nb_comptes") >= S["SHARED_DEVICE"]["nb_comptes_par_appareil_min"])
             .select(F.lit("SHARED_DEVICE").alias("anomaly_type"), F.col("device_id").alias("entity_id"),
                     F.lit("device").alias("entity_type"), "comptes",
                     F.col("nb_comptes").cast("double").alias("metric_value"),
                     F.lit("nb_comptes_par_appareil").alias("metric_name"),
                     F.concat(F.lit("appareil utilise par "), F.col("nb_comptes"),
                              F.lit(" comptes distincts (seuil >= "),
                              F.lit(S["SHARED_DEVICE"]["nb_comptes_par_appareil_min"]), F.lit(")")).alias("motif")))

# 6.2 comptes collecteurs
an_coll = (gf_deg.filter((F.col("inDegree") >= S["COLLECTOR_ACCOUNT"]["nb_emetteurs_distincts_min"])
                         & (F.col("montant_moyen_recu") <= S["COLLECTOR_ACCOUNT"]["montant_moyen_recu_max_mad"]))
           .select(F.lit("COLLECTOR_ACCOUNT").alias("anomaly_type"), F.col("id").alias("entity_id"),
                   F.lit("wallet").alias("entity_type"), F.array("id").alias("comptes"),
                   F.col("inDegree").cast("double").alias("metric_value"),
                   F.lit("nb_emetteurs_distincts").alias("metric_name"),
                   F.concat(F.lit("recoit "), F.col("nb_virements_recus"), F.lit(" virements de "),
                            F.col("inDegree"), F.lit(" comptes, montant moyen "),
                            F.col("montant_moyen_recu"), F.lit(" MAD")).alias("motif")))

# 6.3 cycles
c = S["CIRCULAR_TRANSFER"]
an_cycles = (cycles.filter((F.col("min_nb_tx") >= c["nb_tx_min_par_arete"])
                           & (F.col("min_montant_arete") >= c["montant_min_par_arete_mad"]))
             .select(F.lit("CIRCULAR_TRANSFER").alias("anomaly_type"),
                     F.concat_ws(">", "a", "b", "c", "a").alias("entity_id"),
                     F.lit("cycle").alias("entity_type"), F.array("a", "b", "c").alias("comptes"),
                     F.col("montant_cycle").alias("metric_value"), F.lit("montant_cycle_mad").alias("metric_name"),
                     F.concat(F.lit("cycle A>B>C>A repete "), F.col("min_nb_tx"), F.lit(" fois, "),
                              F.round("montant_cycle", 0), F.lit(" MAD au total")).alias("motif")))

# 6.4 grandes composantes connexes (GraphX)
comp_members = gx.groupBy("component_id").agg(F.first("component_size").alias("component_size"),
                                              F.collect_set("id").alias("comptes"))
an_comp = (comp_members.filter(F.col("component_size") >= S["LARGE_COMPONENT"]["taille_composante_min"])
           .select(F.lit("LARGE_COMPONENT").alias("anomaly_type"),
                   F.concat(F.lit("CC-"), F.col("component_id")).alias("entity_id"),
                   F.lit("component").alias("entity_type"), "comptes",
                   F.col("component_size").cast("double").alias("metric_value"),
                   F.lit("taille_composante").alias("metric_name"),
                   F.concat(F.lit("composante connexe de "), F.col("component_size"),
                            F.lit(" comptes relies par des transferts")).alias("motif")))

anomalies = (an_shared.unionByName(an_coll).unionByName(an_cycles).unionByName(an_comp)
             .withColumn("nb_comptes", F.size("comptes"))
             .withColumn("seuils", F.lit(json.dumps(SEUILS)))
             .withColumn("detected_at", now)
             .cache())
anomalies.groupBy("anomaly_type").count().show()
anomalies.select("anomaly_type", "entity_id", "nb_comptes", "metric_value", "motif").orderBy("anomaly_type", F.desc("metric_value")).show(40, truncate=90)

### 6.5 Verification par rapport aux scenarios injectes par `producer.py` (verite terrain)

In [ ]:
with open(os.path.join(DATA_DIR, "fraud_ground_truth.json"), encoding="utf-8") as f:
    truth = json.load(f)
truth_devices = {d["device_id"] for d in truth["shared_devices"]}
truth_cycles = {tuple(sorted(c)) for c in truth["circular_transfers"]}
truth_collectors = {c["wallet"] for c in truth["collector_accounts"]}
truth_wallets = set()
for d in truth["shared_devices"]: truth_wallets.update(d["wallets"])
for cyc in truth["circular_transfers"]: truth_wallets.update(cyc)
truth_wallets.update(truth_collectors)
legit = truth["legit_atypical_for_false_positive_analysis"]
legit_devices = {k["device_id"] for k in legit["kiosk_devices"]}
legit_wallets = {l["wallet"] for l in legit["landlord_accounts"]}

rows = anomalies.select("anomaly_type", "entity_id", "comptes").collect()
def is_true(r):
    if r.anomaly_type == "SHARED_DEVICE": return r.entity_id in truth_devices
    if r.anomaly_type == "COLLECTOR_ACCOUNT": return r.entity_id in truth_collectors
    if r.anomaly_type == "CIRCULAR_TRANSFER": return tuple(sorted(r.comptes)) in truth_cycles
    return bool(set(r.comptes) & truth_wallets)          # composante contenant un compte frauduleux
flags = {(r.anomaly_type, r.entity_id): is_true(r) for r in rows}

def evaluate(kind, injected):
    detected = [k for k in flags if k[0] == kind]
    tp = sum(flags[k] for k in detected)
    fp = len(detected) - tp
    recall = tp / injected if injected else float("nan")
    return dict(type=kind, injectes=injected, detectes=len(detected), vrais_positifs=tp, faux_positifs=fp,
                rappel=round(recall, 3), precision=round(tp / len(detected), 3) if detected else None)

report = [evaluate("SHARED_DEVICE", len(truth_devices)), evaluate("COLLECTOR_ACCOUNT", len(truth_collectors)),
          evaluate("CIRCULAR_TRANSFER", len(truth_cycles)), evaluate("LARGE_COMPONENT", None)]
for r in report:
    print(r)
fps = [k for k, v in flags.items() if not v]
print("\nFaux positifs (a examiner) :")
for k in fps:
    note = "telephone de teleboutique (legitime connu)" if k[1] in legit_devices else \
           "compte proprietaire/tontine (legitime connu)" if k[1] in legit_wallets else ""
    print("  ", k, note)

### 6.6 Sensibilite des seuils (estimation des faux positifs)

In [ ]:
print("Appareils partages : seuil -> detectes / vrais positifs / faux positifs")
sh = shared.select("device_id", "nb_comptes").collect()
for s_ in (2, 3, 4, 5, 6, 8, 10):
    det = [r for r in sh if r.nb_comptes >= s_]
    tp = sum(r.device_id in truth_devices for r in det)
    print(f"  >= {s_:<3} : {len(det):>5} / {tp} / {len(det) - tp}")

print("Comptes collecteurs (sans / avec critere de montant moyen <= 100 MAD) :")
dg = gf_deg.select("id", "inDegree", "montant_moyen_recu").collect()
for s_ in (5, 10, 15, 20, 30):
    a = [r for r in dg if r.inDegree >= s_]
    b = [r for r in a if r.montant_moyen_recu is not None and r.montant_moyen_recu <= 100]
    print(f"  inDegree >= {s_:<3}: sans critere {len(a):>4} (FP {sum(r.id not in truth_collectors for r in a)})"
          f" | avec critere {len(b):>3} (FP {sum(r.id not in truth_collectors for r in b)})")

cy = cycles.collect()
print(f"Cycles : {len(cy)} cycles bruts, dont {sum(tuple(sorted([r.a, r.b, r.c])) in truth_cycles for r in cy)} injectes ;"
      f" apres seuils (nb_tx >= 2 et montant >= 1000 MAD par arete) : {len([k for k in flags if k[0]=='CIRCULAR_TRANSFER'])}")
print("Composantes connexes : distribution des tailles")
spark.read.parquet(f"{Paths.gold_graphx}/components").groupBy("component_size").count() \
     .orderBy(F.desc("component_size")).show(10)

**Lecture des resultats (a reprendre dans le rapport)**
* *Appareils partages* : les foyers partagent parfois un telephone (2 a 4 comptes) ; au seuil de 5,
  on retrouve les 3 appareils injectes et un telephone de teleboutique legitime (faux positif) ;
  au seuil de 6, plus aucun faux positif sur ce jeu de donnees.
* *Collecteurs* : le degre entrant seul signale aussi les comptes "proprietaire / tontine" (beaucoup
  d'emetteurs mais montants eleves) ; le critere de montant moyen <= 100 MAD les ecarte.
* *Cycles* : des triangles de virements existent naturellement dans les cercles familiaux ; seuls
  les cycles repetes (>= 2 fois par arete) et de montant eleve (>= 1 000 MAD) sont retenus.
* *Composantes connexes* : une composante geante apparait car quelques comptes tres connectes
  (collecteurs, proprietaires) relient de nombreux cercles sociaux. Sa taille signale la presence
  de "hubs" ; elle doit etre analysee avec les degres et le PageRank, pas jugee seule.

## 7. Restitution : exports Parquet, visualisation et metriques du tableau de bord

In [ ]:
truth_udf_rows = [(k[0], k[1], v) for k, v in flags.items()]
truth_df = spark.createDataFrame(truth_udf_rows, "anomaly_type string, entity_id string, scenario_injecte boolean")
anomalies_out = anomalies.join(truth_df, ["anomaly_type", "entity_id"], "left")
(anomalies_out.write.mode("overwrite").partitionBy("anomaly_type").parquet(Paths.gold_graph))
spark.read.parquet(Paths.gold_graph).groupBy("anomaly_type").count().show()

# Motifs bruts aussi conserves pour l'equipe conformite
shared.write.mode("overwrite").parquet(f"{Paths.gold_graph}_motifs/shared_devices")
cycles.write.mode("overwrite").parquet(f"{Paths.gold_graph}_motifs/cycles")
gf_deg.write.mode("overwrite").parquet(f"{Paths.gold_graph}_motifs/degrees_transfers")

In [ ]:
# Metriques pour le tableau de bord conformite (une ligne par execution)
counts_v = {r["type"]: r["count"] for r in vertices.groupBy("type").count().collect()}
counts_e = {r["relationship"]: r["count"] for r in edges.groupBy("relationship").count().collect()}
an_counts = {r["anomaly_type"]: r["count"] for r in anomalies.groupBy("anomaly_type").count().collect()}
comp_stats = spark.read.parquet(f"{Paths.gold_graphx}/components").agg(F.max("component_size")).first()[0]
top_pr = gx.orderBy(F.desc("pagerank")).first()
dashboard = {
    "date_analyse": time.strftime("%Y-%m-%d %H:%M:%S"),
    "nb_wallets": counts_v.get("wallet", 0), "nb_devices": counts_v.get("device", 0),
    "nb_merchants": counts_v.get("merchant", 0),
    "nb_aretes_uses_device": counts_e.get("USES_DEVICE", 0), "nb_aretes_pays": counts_e.get("PAYS", 0),
    "nb_aretes_transfers_to": counts_e.get("TRANSFERS_TO", 0),
    "nb_appareils_partages": an_counts.get("SHARED_DEVICE", 0),
    "nb_comptes_collecteurs": an_counts.get("COLLECTOR_ACCOUNT", 0),
    "nb_cycles_suspects": an_counts.get("CIRCULAR_TRANSFER", 0),
    "nb_grandes_composantes": an_counts.get("LARGE_COMPONENT", 0),
    "taille_max_composante": int(comp_stats or 0),
    "wallet_pagerank_max": top_pr["id"], "pagerank_max": float(top_pr["pagerank"]),
    "nb_comptes_signales_hors_composantes": len({w for r in rows if r.anomaly_type != "LARGE_COMPONENT" for w in r.comptes}),
    "faux_positifs_estimes": len(fps),
}
spark.createDataFrame([dashboard]).write.mode("append").parquet(Paths.gold_graph_dashboard)
with open(os.path.join(LOG_DIR, "part6_dashboard_metrics.json"), "w") as f:
    json.dump({"dashboard": dashboard, "evaluation": report, "seuils": SEUILS, "timings_graphframes": timings,
               "timings_graphx": gx_t}, f, indent=2, ensure_ascii=False)
print(json.dumps(dashboard, indent=2, ensure_ascii=False))

In [ ]:
# Visualisation des sous-graphes suspects : 1 appareil partage, 1 cycle, 1 collecteur (3 panneaux)
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

top_dev = an_shared.orderBy(F.desc("metric_value")).first()
top_cyc = an_cycles.orderBy(F.desc("metric_value")).first()
top_col = an_coll.orderBy(F.desc("metric_value")).first()
color = {"W": "#4C78A8", "D": "#E45756", "M": "#54A24B"}
ecol = {"USES_DEVICE": "#E45756", "TRANSFERS_TO": "#4C78A8", "PAYS": "#54A24B"}

def draw(ax, rows, center, title, layout="star", edge_labels=False):
    G = nx.DiGraph()
    for e in rows:
        G.add_edge(e.src, e.dst, rel=e.relationship, n=e.nb_tx, m=e.montant_total_mad)
    if not G.nodes:
        ax.set_title(title + " (aucun)"); ax.axis("off"); return
    if layout == "star":
        others = [n for n in G.nodes if n != center]
        pos = nx.shell_layout(G, nlist=[[center], others])
    else:
        pos = nx.circular_layout(G)
    big = {center} if layout == "star" else set(G.nodes)
    nx.draw_networkx_nodes(G, pos, node_color=[color[n[0]] for n in G.nodes],
                           node_size=[1400 if n in big else 220 for n in G.nodes], alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, labels={n: n for n in big}, font_size=7, font_color="white", ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color=[ecol[d["rel"]] for _, _, d in G.edges(data=True)],
                           width=[1 + min(d["n"], 4) for _, _, d in G.edges(data=True)],
                           arrowsize=14, node_size=1400 if layout != "star" else 300,
                           connectionstyle="arc3,rad=0.12", alpha=0.75, ax=ax)
    if edge_labels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels={(u, v): f"{d['n']} tx\n{d['m']:,.0f} MAD"
                                                          for u, v, d in G.edges(data=True)},
                                     font_size=8, label_pos=0.5, ax=ax)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
if top_dev:
    draw(axes[0], edges.filter(F.col("dst") == top_dev.entity_id).collect(), top_dev.entity_id,
         f"Appareil partage {top_dev.entity_id} : {int(top_dev.metric_value)} comptes (USES_DEVICE)")
if top_cyc:
    a, b, c_ = top_cyc.comptes
    draw(axes[1], edges.filter((F.col("relationship") == "TRANSFERS_TO") & F.col("src").isin(a, b, c_)
                               & F.col("dst").isin(a, b, c_)).collect(), a,
         "Transfert circulaire A -> B -> C -> A", layout="circle", edge_labels=True)
if top_col:
    draw(axes[2], edges.filter((F.col("relationship") == "TRANSFERS_TO") & (F.col("dst") == top_col.entity_id)).collect(),
         top_col.entity_id, f"Compte collecteur {top_col.entity_id} : {int(top_col.metric_value)} emetteurs, petits montants")
fig.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor=color["W"], markersize=11, label="wallet"),
                    Line2D([0], [0], marker="o", color="w", markerfacecolor=color["D"], markersize=11, label="device"),
                    Line2D([0], [0], color=ecol["USES_DEVICE"], lw=2, label="USES_DEVICE"),
                    Line2D([0], [0], color=ecol["TRANSFERS_TO"], lw=2, label="TRANSFERS_TO")],
           loc="lower center", ncol=4, frameon=False)
fig.suptitle("SahlPay - sous-graphes signales a l'equipe conformite (structures inhabituelles, pas une preuve de fraude)",
             fontsize=13)
out_png = os.path.join(PROJECT_DIR, "rapport", "captures", "part6_sous_graphe_suspect.png")
os.makedirs(os.path.dirname(out_png), exist_ok=True)
fig.savefig(out_png, dpi=120, bbox_inches="tight")
print("visualisation enregistree :", out_png)
plt.show()

In [ ]:
print(timings)
spark.stop()